# 03. Behavior Cloning 学習 (Google Colab)

ローカルで収集した `data/episodes/`(各エピソードの `video.mp4` + `actions.csv`)と `data/manifest.csv` を使って、
模倣学習 (Behavior Cloning) でCNNを学習します。このノートブックは **Google Colab** にアップロードして実行してください。

## 事前準備 (ローカルで実行)

`data` フォルダをzip化し、Googleドライブの適当なフォルダ (例: `MyDrive/mario_bc/`) にアップロードしてください。

Windows (PowerShell):

```powershell
Compress-Archive -Path data -DestinationPath mario_data.zip
```

Ubuntu (bash、`zip` が無ければ `sudo apt install zip`):

```bash
zip -r mario_data.zip data
```

## Colabでの実行手順

1. 「ランタイム」→「ランタイムのタイプを変更」→ ハードウェアアクセラレータを **GPU** に設定
2. 以下のセルを上から順に実行(まずは2〜3エピソードだけで1epochの試験実行をしてエラーが出ないことを確認してから、本学習に進むことを推奨)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/mario_bc/mario_data.zip"  # change this to wherever you uploaded the zip
EXTRACT_DIR = Path("/content/mario_data")

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(EXTRACT_DIR)

DATA_DIR = EXTRACT_DIR / "data"
EPISODES_DIR = DATA_DIR / "episodes"
MANIFEST_PATH = DATA_DIR / "manifest.csv"

episode_dirs_preview = sorted(EPISODES_DIR.glob("ep_*"))
print(f"{len(episode_dirs_preview)} episodes found")
print(episode_dirs_preview[:5])

In [ ]:
# A verbatim copy of SIMPLE_MOVEMENT from gym_super_mario_bros==9.1.0.
# Colab does not need the full nes-py stack to train, so only the action list
# lives here. It must stay identical to the local
# gym_super_mario_bros.actions.SIMPLE_MOVEMENT (v9.1.0).
SIMPLE_MOVEMENT = [
    ["NOOP"],
    ["right"],
    ["right", "A"],
    ["right", "B"],
    ["right", "A", "B"],
    ["A"],
    ["left"],
]
print(SIMPLE_MOVEMENT)

In [ ]:
# --- Keep this block identical in 03_train_bc.ipynb and 04_playagent.ipynb ---
from collections import deque

import cv2
import numpy as np
import torch
import torch.nn as nn

FRAME_SIZE = (84, 84)

# --- Exercise knob: how many consecutive frames the model sees at once -----
# 1 turns this into an ordinary single-image classifier; 4 is the usual choice
# in the Atari literature. Try values in the 1-4 range and compare how well the
# agent actually plays.
#
# A single frame cannot show velocity. Whether Mario is rising or falling in a
# jump, and how fast he is moving, only becomes visible across several frames.
# Stacking more frames makes that observable, at the cost of a larger input and
# more memory (the dataset holds STACK_SIZE copies of every frame).
#
# Set this in 03 only. 04_playagent.ipynb reads the value back out of the
# checkpoint, so there is nothing to keep in sync there by hand.
STACK_SIZE = 4
# ---------------------------------------------------------------------------

NUM_ACTIONS = len(SIMPLE_MOVEMENT)


def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    """Convert an RGB frame to grayscale and resize it to 84x84."""
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    return cv2.resize(gray, FRAME_SIZE, interpolation=cv2.INTER_AREA)


class FrameStacker:
    """Stack the most recent STACK_SIZE preprocessed frames."""

    def __init__(self, stack_size: int = STACK_SIZE):
        self.stack_size = stack_size
        self.frames = deque(maxlen=stack_size)

    def reset(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.clear()
        for _ in range(self.stack_size):
            self.frames.append(processed_frame)
        return self.get()

    def step(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.append(processed_frame)
        return self.get()

    def get(self) -> np.ndarray:
        return np.stack(self.frames, axis=0)


class MarioCNN(nn.Module):
    def __init__(self, num_actions: int = NUM_ACTIONS, in_channels: int = STACK_SIZE):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, *FRAME_SIZE)
            n_flatten = self.features(dummy).shape[1]
        self.head = nn.Sequential(
            nn.Linear(n_flatten, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.float() / 255.0
        x = self.features(x)
        return self.head(x)
# --- end of shared block ---

## エピソード冒頭の NOOP を除外する

人間はエピソード開始後、キーに手を伸ばすまでに少し間があるため、**各エピソードの先頭には必ず
NOOP のラベルが並びます**。この区間の画像はどれも「ステージ開始位置に立ったままのマリオ」で、
最初に `right` を押したフレームとも画素単位でほぼ完全に同一です。

これをそのまま学習させると、**同じ画像が NOOP と `right` の両方のラベルで矛盾して与えられ、
しかも NOOP のほうが圧倒的に多数派**になります (手元のデータでは約24対1)。
結果としてモデルは開始画面で NOOP を返すようになり、NOOP では画面が変わらないので
次も同じ入力・同じ出力となり、**マリオが一歩も動かないまま固まります**。

`TRIM_LEADING_NOOP = True` で、各エピソードの先頭にある NOOP の連続を除外します。
捨てるのは全体の2%程度で、クラス分布はほとんど変わりません。

なお、**エピソード途中の NOOP は残します**。着地待ちや敵をやり過ごす場面など、
静止していること自体が正しい判断であるケースが含まれるためです。
除外するのは「まだプレイを始めていない」冒頭部分だけです。


In [ ]:
import pandas as pd

NOOP_ACTION = SIMPLE_MOVEMENT.index(["NOOP"])
TRIM_LEADING_NOOP = True


def list_episode_dirs(episodes_dir: Path):
    return sorted(p for p in episodes_dir.glob("ep_*") if p.is_dir())


def trim_leading_noop(frames, action_list):
    """Drop the frames at the start of an episode, before the player first moved.

    A human takes a moment to reach for the keys after the episode starts, so
    every recording opens with a run of NOOP labels on what is, pixel for pixel,
    the same start-of-level image. Left in, that one image reaches the model
    roughly 24 times labelled NOOP for every once it is labelled "right", so the
    model learns to do nothing at the start. NOOP does not change the screen, so
    the next input is identical, the answer is NOOP again, and the agent stays
    frozen for the whole episode. Trimming the run leaves the start frame
    carrying the action the player actually took.
    """
    start = 0
    while start < len(action_list) and action_list[start] == NOOP_ACTION:
        start += 1
    return frames[start:], action_list[start:], start


def load_episode(ep_dir: Path):
    """Load an episode's video + CSV and return preprocessed frames and actions."""
    actions_df = pd.read_csv(ep_dir / "actions.csv")
    action_list = actions_df.sort_values("frame_idx")["action"].tolist()

    frames = []
    cap = cv2.VideoCapture(str(ep_dir / "video.mp4"))
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frames.append(preprocess_frame(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)))
    cap.release()

    # Fall back to the shorter of the two in case the video and CSV lengths differ
    n = min(len(frames), len(action_list))
    frames, action_list = frames[:n], action_list[:n]

    trimmed = 0
    if TRIM_LEADING_NOOP:
        frames, action_list, trimmed = trim_leading_noop(frames, action_list)
    return frames, action_list, trimmed


class MarioEpisodeDataset(torch.utils.data.Dataset):
    def __init__(self, episode_dirs):
        self.stacks = []
        self.labels = []
        total_trimmed = 0
        for ep_dir in episode_dirs:
            frames, action_list, trimmed = load_episode(ep_dir)
            total_trimmed += trimmed
            if not frames:
                print(f"  {ep_dir.name}: nothing left after trimming - skipped")
                continue
            # Pass STACK_SIZE explicitly rather than relying on the default that
            # was bound when FrameStacker was defined.
            stacker = FrameStacker(STACK_SIZE)
            self.stacks.append(stacker.reset(frames[0]))
            self.labels.append(action_list[0])
            for frame, action in zip(frames[1:], action_list[1:]):
                self.stacks.append(stacker.step(frame))
                self.labels.append(action)

        if TRIM_LEADING_NOOP:
            print(f"  trimmed {total_trimmed} leading NOOP frames from {len(episode_dirs)} episodes")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        stack = torch.from_numpy(self.stacks[idx].copy())
        return stack, self.labels[idx]

## train/val 分割 (エピソード単位)

フレーム単位で分割すると隣接フレームがtrain/valに漏れて検証精度が過大評価されるため、エピソード単位で分割する。

In [ ]:
import random

episode_dirs = list_episode_dirs(EPISODES_DIR)
random.seed(0)
shuffled = episode_dirs[:]
random.shuffle(shuffled)

split_idx = max(1, int(len(shuffled) * 0.85))
train_dirs = shuffled[:split_idx]
val_dirs = shuffled[split_idx:] or shuffled[-1:]

print(f"train episodes: {len(train_dirs)}, val episodes: {len(val_dirs)}")

train_dataset = MarioEpisodeDataset(train_dirs)
val_dataset = MarioEpisodeDataset(val_dirs)
print(f"train samples: {len(train_dataset)}, val samples: {len(val_dataset)}")

## クラス重みの計算

`02_review_data.ipynb` で確認したように "right" 等の多数派アクションにサンプルが偏っているため、出現頻度の逆数でクラス重み付けする。

In [ ]:
from collections import Counter

label_counts = Counter(train_dataset.labels)
class_weights = torch.tensor(
    [1.0 / label_counts.get(i, 1) for i in range(NUM_ACTIONS)],
    dtype=torch.float32,
)
class_weights = class_weights / class_weights.sum() * NUM_ACTIONS

print("samples per class:", dict(sorted(label_counts.items())))
print("class weights:", class_weights.tolist())

## 学習

動作確認は `EPOCHS = 1` かつ少数エピソードで、本学習は `EPOCHS = 15` 程度から調整してください。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EPOCHS = 15
BATCH_SIZE = 64
LEARNING_RATE = 2.5e-4

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = MarioCNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for stacks, labels in train_loader:
        stacks, labels = stacks.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(stacks)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    train_loss = total_loss / total
    train_acc = correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for stacks, labels in val_loader:
            stacks, labels = stacks.to(device), labels.to(device)
            logits = model(stacks)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total if val_total else float("nan")

    print(
        f"epoch {epoch:3d}/{EPOCHS}  train_loss={train_loss:.4f}  "
        f"train_acc={train_acc:.3f}  val_acc={val_acc:.3f}"
    )

## 混同行列 (検証データ)

7クラス中の多数派アクションに引っ張られて全体精度だけでは判断しにくいため、クラスごとの予測傾向も確認する。

In [ ]:
confusion = np.zeros((NUM_ACTIONS, NUM_ACTIONS), dtype=int)
model.eval()
with torch.no_grad():
    for stacks, labels in val_loader:
        preds = model(stacks.to(device)).argmax(dim=1).cpu().numpy()
        for t, p in zip(labels.numpy(), preds):
            confusion[t, p] += 1

action_names = ["_".join(a) if a else "NOOP" for a in SIMPLE_MOVEMENT]
pd.DataFrame(confusion, index=action_names, columns=action_names)

## モデルの書き出し

In [ ]:
import time as _time

from google.colab import files

# The stack size is baked into the file name and stored inside the checkpoint,
# so several runs with different STACK_SIZE values can sit side by side in
# models/checkpoints/ and 04 can rebuild the matching model on its own.
CHECKPOINT_NAME = f"mario_bc_s{STACK_SIZE}_{_time.strftime('%Y%m%d_%H%M%S')}.pt"
torch.save(
    {
        "state_dict": model.state_dict(),
        "stack_size": STACK_SIZE,
        "frame_size": FRAME_SIZE,
        "num_actions": NUM_ACTIONS,
    },
    CHECKPOINT_NAME,
)
print(f"saved: {CHECKPOINT_NAME} (STACK_SIZE={STACK_SIZE})")

files.download(CHECKPOINT_NAME)

ダウンロードした `.pt` ファイルをローカルの `models/checkpoints/` に配置してください。`04_playagent.ipynb` で使用します。

## 演習: `STACK_SIZE` を変えて比較する

`STACK_SIZE` を **1〜4** の範囲で変えて学習し、`04_playagent.ipynb` でプレイさせて結果を比べてください。
ファイル名に `s1` / `s4` のように値が入るので、複数のチェックポイントを並べて置けます。

| STACK_SIZE | 入力 | val_acc | 平均 x_pos_max | クリア率 |
|---|---|---|---|---|
| 1 | `[1, 84, 84]` (普通の画像分類) | | | |
| 2 | `[2, 84, 84]` | | | |
| 4 | `[4, 84, 84]` (既定値) | | | |

`STACK_SIZE = 1` は「1枚の静止画からアクションを推論する、普通の画像分類モデル」になります。
1枚の絵からは速度が読めない(ジャンプの上昇中か落下中か、走っているか止まっているかが判別できない)ため、
特にジャンプの制御が苦しくなるはずです。混同行列でジャンプ系 (`right_A`, `A`, `right_A_B`) の行が
どう混ざるかを見てください。

### 判断は val_acc ではなくプレイ結果で

このタスクでは **検証精度と実際のプレイのうまさはあまり一致しません。**

- 人間のプレイには「押しても押さなくても大差ない場面」が大量にあり、そこでの不一致は
  val_acc を下げますが、プレイの質はほとんど落ちません。
- 逆に着地の1フレームでジャンプを押し損ねると穴に落ちて即終了します。
  val_acc への影響は 0.1% 未満でも、到達距離は半分になります。
- BC には誤差が累積する性質 (compounding error) があります。1回ずれると人間のプレイデータに無い
  状況に入り、そこでの予測はさらに外れます。val_acc は「人間の軌跡上での正解率」しか測っていないので、
  この影響を捉えられません。

**結論は必ず `04` の到達距離 (`x_pos_max`、ゴールは 3161) とクリア率で出してください。**
1回の実行では運の影響が大きいので、`EPISODES` は10以上を推奨します。
